In [1]:
import sys
!{sys.executable} -m pip install numpy
import sys
!{sys.executable} -m pip install python-sat
import sys
!{sys.executable} -m pip install dwave-neal

In [2]:
import numpy as np
from itertools import product
import random
import sys
from pysat.solvers import Minisat22

np.set_printoptions(
    threshold=sys.maxsize, # 전체 출력
    linewidth=150,        # 한 줄 길이를 넉넉하게
    precision=3,          # 소수점 3자리까지
    suppress=True         # 0.000001을 0.으로 표시
)

In [3]:
# [수정 2.5] 연속 계수 → 이산 계수 (Pelofske 2024, Section 2.1)
#   - 기존: coeff_boundary = 1, np.random.uniform(-1, 1) 연속 균일분포
#   - 수정: lin2 {-1, +1} 또는 lin20 {-1.0, -0.9, ..., 0.9, 1.0} 이산 집합
#   - lin2가 SA에 대해 가장 어려운 문제를 생성 (동일 에너지 상태가 많아 축퇴 증가)
COEFF_LIN2 = [-1, 1]                                              # [수정 2.5]
COEFF_LIN20 = [round(-1 + 0.1 * i, 1) for i in range(21)]        # [수정 2.5]

print("lin2:", COEFF_LIN2)
print("lin20:", COEFF_LIN20)

lin2: [-1, 1]
lin20: [-1.0, -0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


## gen_random_qubo
n x n 크기의 랜덤 QUBO 생성

[수정 2.5] `coeff_type` 파라미터 추가
- `'lin2'`: {-1, +1} — 가장 어려움 (축퇴 많음)
- `'lin20'`: {-1.0, -0.9, ..., 0.9, 1.0} — 중간
- 기존 `np.random.uniform(-1, 1)` 연속 균일분포 제거

In [4]:
# [수정 2.5] coeff_type 파라미터 추가
#   - 기존: np.random.uniform(-coeff_boundary, coeff_boundary, (n, n)) 연속 균일분포
#   - 수정: 이산 계수 집합에서 랜덤 선택 (논문 방식)
def gen_random_qubo(n, coeff_type='lin2'):                        # [수정 2.5] coeff_type 추가
    coeffs = COEFF_LIN2 if coeff_type == 'lin2' else COEFF_LIN20  # [수정 2.5] 이산 계수 선택
    random_qubo = np.array([[random.choice(coeffs) for _ in range(n)] for _ in range(n)])  # [수정 2.5]
    return np.triu(random_qubo)  # [수정] 상삼각만 유지, 하삼각은 0

In [5]:
print("lin2:")
print(gen_random_qubo(5, 'lin2'))       # {-1, +1}만 나옴
print("\nlin20:")
print(gen_random_qubo(5, 'lin20'))      # {-1.0, -0.9, ..., 1.0} 중에서 나옴

lin2:
[[ 1  1 -1 -1 -1]
 [ 0 -1  1 -1 -1]
 [ 0  0  1 -1 -1]
 [ 0  0  0  1 -1]
 [ 0  0  0  0 -1]]

lin20:
[[-0.9 -0.4  0.2 -0.8 -0.5]
 [ 0.  -0.7 -0.3  0.1  0.4]
 [ 0.   0.   1.   0.2 -0.4]
 [ 0.   0.   0.   0.1 -0.8]
 [ 0.   0.   0.   0.   0.3]]


## find_opt_brute_force
brute force로 최적해, 최적 값, 축퇴도 탐색

In [6]:
# matrix mat를 입력 받아, 최적해, 최적 값, 축퇴도를 출력
# [수정] num_degenerate 반환 추가 — brute force로 동일 에너지 상태 개수를 세서 유일성 검증
def find_opt_brute_force(mat, debug=False):
    n = mat.shape[0]

    best_x = None
    best_val = float('inf')
    num_degenerate = 0  # [수정] 축퇴도 카운트 추가

    for bits in product([0,1], repeat=n):
        x = np.array(bits)
        cur_val = x@mat@x # 스칼라값

        if cur_val < best_val - 1e-12:
            best_val = cur_val
            best_x = x
            num_degenerate = 1
        elif abs(cur_val - best_val) < 1e-12:
            num_degenerate += 1

    if debug: print("opt_x:", best_x, "\nopt_val:", best_val, "\ndegeneracy:", num_degenerate)
        
    return best_x, best_val, num_degenerate

## gen_concatenated_random_qubo
subgraph 균등 분할 → concat하여 큰 QUBO 생성

[수정] 논문 방식으로 균등 분할 (Pelofske 2024, Section 2.2)
- 기존: `min_sub_graph_size`, `max_sub_graph_size`로 랜덤 크기 분할
- 수정: `max_sub_graph_size`만 사용, 균등 분할 (크기 차이 최대 1)
- 논문: "identically sized, unless there are any odd divisions in which case ... different by at most 1 variable"

In [7]:
# [수정] 논문 방식 균등 분할 (Pelofske 2024, Section 2.2)
#   - 기존: min_sub_graph_size ~ max_sub_graph_size 랜덤 크기
#   - 수정: max_sub_graph_size 기준 균등 분할 (크기 차이 최대 1)
#   예) n=10, max_sub_graph_size=3 → k=4개 partition → 크기 [3, 3, 2, 2]
def gen_concatenated_random_qubo(n, max_sub_graph_size, coeff_type='lin2', debug=False):  # [수정] min 제거
    # 균등 분할: k개 partition, 각 크기 차이 최대 1                # [수정]
    k = max(1, -(-n // max_sub_graph_size))  # ceil(n / max_size)  # [수정]

    mat = np.zeros((n, n))
    opt = np.zeros(n)
    cur_size = 0

    for i in range(k):                                             # [수정]
        start = i * n // k                                         # [수정]
        end = (i + 1) * n // k                                     # [수정]
        cur_n = end - start                                        # [수정]

        cur_mat = gen_random_qubo(cur_n, coeff_type)
        cur_opt, _, _ = find_opt_brute_force(cur_mat)

        mat[start:end, start:end] = cur_mat                        # [수정] cur_size → start:end
        opt[start:end] = cur_opt                                   # [수정]

        if debug:
            print(f"  partition {i}: vars [{start}, {end}), size={cur_n}")

    if debug:
        print(f"  총 {k}개 partition, 크기: {[((i+1)*n//k - i*n//k) for i in range(k)]}")
        print()

    return mat, opt

In [8]:
gen_concatenated_random_qubo(10, 5, debug=True) # n=10, max_sub_graph_size=5 → 균등 분할

  partition 0: vars [0, 5), size=5
  partition 1: vars [5, 10), size=5
  총 2개 partition, 크기: [5, 5]



(array([[-1.,  1.,  1.,  1., -1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  1., -1.,  1.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  1.,  1.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  1., -1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0., -1., -1.,  1.,  1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0., -1., -1., -1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  1., -1., -1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.]]),
 array([1., 0., 0., 0., 1., 0., 1., 1., 1., 0.]))

## posiform_planting
posiform planting (MiniSat uniqueness 검증)

[수정 2.4] 별도의 posiform 전용 행렬(`mat_posiform`)에 항을 쌓아서 반환.
- 기존: `mat`(random QUBO)를 직접 수정 → α=1 고정
- 수정: 빈 행렬을 만들어 posiform 항만 쌓고 반환 → `gen_posiform_qubo`에서 `Q_final = Q_random + α × Q_posiform`으로 결합

In [21]:
# [수정 2.4] posiform을 별도 행렬에 쌓아서 반환
#   - 기존: posiform_planting(mat, opt, weight) → mat를 직접 수정 (void)
#   - 수정: posiform_planting(opt, n, weight) → mat_posiform을 생성하여 반환
#   → gen_posiform_qubo에서 Q_final = Q_random + α × Q_posiform 으로 결합
#
# [수정] 논문(Hahn 2023, Section 2.3) 방식:
#   - 3개 wrong tuple 중 1개만 랜덤 선택하여 추가
# [수정] MiniSat으로 2-SAT uniqueness 검증:
#   - 유일해질 때까지 clause를 계속 추가 (2-SAT phase transition은 O(n))
# [수정] 상삼각 행렬 형식 유지:
#   - off-diagonal 업데이트 시 mat[min(i,j)][max(i,j)]에 접근
def posiform_planting(opt, n, weight):         # [수정 2.4] mat 제거, n 추가, 반환값 있음
    mat_posiform = np.zeros((n, n))            # [수정 2.4] 별도의 posiform 전용 행렬
    all_tuples = [(0, 0), (0, 1), (1, 0), (1, 1)]

    # CNF clauses: MiniSat uniqueness 검증에 사용
    clauses_cnf = []

    # 상한: 2-SAT phase transition은 O(n)이므로 10*n이면 충분 - (논문에 나온건 아니다)
    max_clauses = 10 * n

    def add_posiform_term(i, j):
        """pair (i,j)에 대해 wrong tuple 1개를 랜덤 선택하여 posiform 항 + CNF clause 추가"""
        target_tuple = (int(opt[i]), int(opt[j]))

        # target을 제외한 3개 wrong tuple
        wrong_tuples = []
        for t in all_tuples:
            if t != target_tuple:
                wrong_tuples.append(t)

        wi, wj = random.choice(wrong_tuples)

        # [수정] 상삼각 형식 유지: off-diagonal은 항상 mat[작은][큰]에 접근
        lo, hi = min(i, j), max(i, j)

        # [수정 2.4] mat → mat_posiform: posiform 전용 행렬에 기록
        if wi == 0 and wj == 0:   # (1-xi)(1-xj) * weight
            mat_posiform[i][i] -= weight
            mat_posiform[j][j] -= weight
            mat_posiform[lo][hi] += weight
        elif wi == 0 and wj == 1: # (1-xi)xj * weight
            mat_posiform[j][j] += weight
            mat_posiform[lo][hi] -= weight
        elif wi == 1 and wj == 0: # xi(1-xj) * weight
            mat_posiform[i][i] += weight
            mat_posiform[lo][hi] -= weight
        else:                     # xixj * weight
            mat_posiform[lo][hi] += weight

        # CNF clause 추가: wrong tuple (wi,wj) 배제
        # CNF clause자체는 잘 이해가 안됨
        lit_i = (i + 1) if wi == 0 else -(i + 1)
        lit_j = (j + 1) if wj == 0 else -(j + 1)
        clauses_cnf.append([lit_i, lit_j])

    def check_uniqueness():
        """MiniSat으로 target 외 다른 해가 있는지 확인."""
        with Minisat22() as solver:
            for clause in clauses_cnf:
                solver.add_clause(clause)

            # target 차단: "적어도 하나의 변수가 target과 달라야 한다"
            blocking = []
            for i in range(n):
                if int(opt[i]) == 1:
                    blocking.append(-(i + 1))
                else:
                    blocking.append(i + 1)

            solver.add_clause(blocking)
            return not solver.solve()

    check_interval = max(1, n // 4)

    for step in range(max_clauses):
        i, j = random.sample(range(n), 2)
        add_posiform_term(i, j)

        if (step + 1) % check_interval == 0:
            if check_uniqueness():
                print(f"  [posiform] {step + 1} clauses로 유일성 확보")
                return mat_posiform          # [수정 2.4] 별도 행렬 반환

    print(f"  [Warning] {max_clauses} clauses 후에도 유일성 미확보")
    return mat_posiform                      # [수정 2.4] 별도 행렬 반환

## gen_posiform_qubo
random QUBO + posiform planting 결합

[수정 2.4] `posiform_scale`(α) 파라미터 추가 (Pelofske 2024, Section 2.1)
- `Q_final = Q_random + α × Q_posiform`
- α가 클수록(1.0) posiform 신호가 강해서 SA가 쉽게 풀 수 있음
- α가 작을수록(0.01) random QUBO가 지배적 → SA가 어려워짐
- 논문 실험값: α = 0.1, 0.01 (0.01이 가장 어려움)

In [22]:
# [수정 2.4] posiform_scale(α) 파라미터 추가
#   - 기존: posiform이 random QUBO에 직접 더해짐 (α=1 고정)
#   - 수정: Q_final = Q_random + α × Q_posiform
def gen_posiform_qubo(n, max_sub_graph_size, weight, posiform_scale=1.0, coeff_type='lin2'):  # [수정] min 제거
    # 1단계: random QUBO 생성 (block-diagonal, 균등 분할)
    mat_random, opt = gen_concatenated_random_qubo(n, max_sub_graph_size, coeff_type)  # [수정] min 제거

    # 2단계: posiform QUBO를 별도 행렬로 생성                    # [수정 2.4]
    mat_posiform = posiform_planting(opt, n, weight)              # [수정 2.4]

    # 3단계: 결합 — Q_final = Q_random + α × Q_posiform          # [수정 2.4]
    mat = mat_random + posiform_scale * mat_posiform              # [수정 2.4]

    opt_val = opt @ mat @ opt

    print(f"  [에너지 분해] E_random={opt @ mat_random @ opt:.4f}, "
          f"α·E_posiform={posiform_scale * (opt @ mat_posiform @ opt):.4f}, "
          f"E_total={opt_val:.4f}")

    return mat, opt, opt_val

In [23]:
# [수정] 검증 루프: gen_posiform_qubo로 생성 → brute force로 GS 검증
chk = True
cnt = 0
while chk and cnt < 1000:
    cnt += 1
    if(cnt %50 == 0): print(cnt)

    qubo, opt_init, _ = gen_posiform_qubo(10, 5, weight=1, posiform_scale=0.1)  # [수정] min 제거

    opt, opt_val, deg = find_opt_brute_force(qubo, False)
    if not np.array_equal(opt_init, opt):
        print("[Error] Optimum is changed!!!")
        chk = False
    if deg > 1:
        print(f"[Warning] Degenerate ground state: {deg} solutions (cnt={cnt})")

print("Done!" if chk else "Failed!")

  [posiform] 24 clauses로 유일성 확보
  [에너지 분해] E_random=-8.0000, α·E_posiform=-0.9000, E_total=-8.9000
  [posiform] 20 clauses로 유일성 확보
  [에너지 분해] E_random=-7.0000, α·E_posiform=-0.4000, E_total=-7.4000
  [posiform] 52 clauses로 유일성 확보
  [에너지 분해] E_random=-8.0000, α·E_posiform=-1.8000, E_total=-9.8000
  [posiform] 46 clauses로 유일성 확보
  [에너지 분해] E_random=-7.0000, α·E_posiform=-1.7000, E_total=-8.7000
  [posiform] 50 clauses로 유일성 확보
  [에너지 분해] E_random=-4.0000, α·E_posiform=-1.4000, E_total=-5.4000
  [posiform] 30 clauses로 유일성 확보
  [에너지 분해] E_random=-3.0000, α·E_posiform=-0.2000, E_total=-3.2000
  [posiform] 56 clauses로 유일성 확보
  [에너지 분해] E_random=-9.0000, α·E_posiform=-1.7000, E_total=-10.7000
  [posiform] 36 clauses로 유일성 확보
  [에너지 분해] E_random=-6.0000, α·E_posiform=-0.9000, E_total=-6.9000
  [posiform] 34 clauses로 유일성 확보
  [에너지 분해] E_random=-8.0000, α·E_posiform=-0.7000, E_total=-8.7000
  [posiform] 56 clauses로 유일성 확보
  [에너지 분해] E_random=-6.0000, α·E_posiform=-0.8000, E_total=-6.8000
  [posifo

In [24]:
n = 10          # 변수 개수
max_sub_graph_size = 5  # [수정] min 제거, 균등 분할 (논문 방식) 나누어 떨어지지 않으면 최대 1개 차이
weight = 1      # posiform clause 개별 계수 (Pelofske 2024: "All posiform coefficients are chosen as 1")

# [수정 2.4] posiform_scale(α) 추가
posiform_scale = 0.1                                              # [수정 2.4]

# [수정 2.5] coeff_type 추가
# 'lin2'  → {-1, +1} — 가장 어려움 (축퇴 많음)
# 'lin20' → {-1.0, -0.9, ..., 1.0} — 중간
coeff_type = 'lin2' # or 'lin20'                                  # [수정 2.5]

qubo, opt_x, opt_val = gen_posiform_qubo(n, max_sub_graph_size, weight, posiform_scale, coeff_type)

# Cross-block: Q_random이 0 + α × posiform → posiform clause가 걸린 쌍만 non-zero, 나머지는 0
print(f"\nposiform planted qubo (α={posiform_scale}, coeff={coeff_type}):\n", qubo)
print()
print("opt_x:", opt_x)
print("opt_val:", opt_val)

  [posiform] 46 clauses로 유일성 확보
  [에너지 분해] E_random=-5.0000, α·E_posiform=-1.0000, E_total=-6.0000

posiform planted qubo (α=0.1, coeff=lin2):
 [[ 1.  -0.9  1.1  1.1  1.   0.   0.   0.   0.  -0.1]
 [ 0.   1.   0.9 -1.  -1.2  0.3  0.1  0.   0.1  0.1]
 [ 0.   0.  -1.  -1.   0.9  0.   0.   0.1  0.  -0.1]
 [ 0.   0.   0.  -1.2  1.   0.1  0.   0.   0.  -0.3]
 [ 0.   0.   0.   0.   1.2  0.2  0.  -0.2 -0.1  0. ]
 [ 0.   0.   0.   0.   0.  -1.3  1.1 -1.  -1.1  0.9]
 [ 0.   0.   0.   0.   0.   0.   1.2 -0.9 -1.   1. ]
 [ 0.   0.   0.   0.   0.   0.   0.   1.2  1.   1.1]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   1.1 -1.2]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0.  -0.9]]

opt_x: [0. 0. 1. 1. 0. 1. 0. 0. 1. 1.]
opt_val: -6.0


## SA (Simulated Annealing) 실험
posiform planted QUBO를 여러 인스턴스 생성하여 SA로 풀고 통계 집계.

**하이퍼파라미터:**
- `num_instances`: QUBO 인스턴스 수 (각각 독립 생성)
- `num_reads`: 인스턴스당 SA sample 수
- `num_sweeps`: SA sweep 수 (클수록 정확하지만 느림)
- QUBO 생성 파라미터: `n`, `max_sub_graph_size`, `a`, `posiform_scale`, `coeff_type`

In [42]:
import neal
import time

# ─── 하이퍼파라미터 (조정 가능) ───
num_instances = 10    # QUBO 인스턴스 수
num_reads = 100       # 인스턴스당 SA sample 수
num_sweeps = 5000     # SA sweep 수 (논문: 1~10000 범위에서 S-curve 전이)

# QUBO 생성 파라미터 (위 실행 셀과 독립적으로 설정 가능)
sa_n = 500 # n size
sa_max_sub = 14 # subgraph size
sa_weight = 1 # posiform qubo 계수
sa_alpha = 0  # posiform_scale(α)
sa_coeff = 'lin20'

# ─── numpy 행렬 → dict 변환 (neal 입력 형식) ───
def matrix_to_qubo_dict(mat):
    Q = {}
    n = mat.shape[0]
    for i in range(n):
        for j in range(i, n):
            if mat[i][j] != 0:
                Q[(i, j)] = float(mat[i][j])
    return Q

# SA 실험
# 정답 판정: SA가 찾은 bitstring == GS bitstring
# posiform planting + MiniSat uniqueness 검증으로
# target이 유일한 ground state(GS)임이 보장되므로
# bitstring 일치 = GS를 찾음 (부동소수점 오차 없는 정확한 판정)
sampler = neal.SimulatedAnnealingSampler()
total_correct = 0
total_samples = 0
total_hamming = 0

print(f"═══ SA 실험 (instances={num_instances}, reads={num_reads}, sweeps={num_sweeps}) ═══")
print(f"  QUBO: n={sa_n}, max_sub={sa_max_sub}, α={sa_alpha}, coeff={sa_coeff}")
print()

t0 = time.time()
for inst in range(num_instances):
    # 매 인스턴스마다 새로운 QUBO 생성 (랜덤 seed 다름)
    qubo_sa, opt_sa, opt_val_sa = gen_posiform_qubo(
        sa_n, sa_max_sub, sa_weight, sa_alpha, sa_coeff)

    # planted solution의 bitstring (유일한 GS)
    target_bitstring = ''.join(str(int(x)) for x in opt_sa)
    Q_dict = matrix_to_qubo_dict(qubo_sa)

    ss = sampler.sample_qubo(Q_dict, num_reads=num_reads, num_sweeps=num_sweeps)

    inst_correct = 0
    inst_hamming = 0
    for sample, energy, _ in ss.data(['sample', 'energy', 'num_occurrences']):
        found = ''.join(str(sample[k]) for k in range(sa_n))
        hd = sum(1 for a, b in zip(target_bitstring, found) if a != b)
        inst_hamming += hd
        total_samples += 1
        if found == target_bitstring:  # bitstring 일치 판정
            inst_correct += 1
            total_correct += 1
        total_hamming += hd

    inst_rate = 100.0 * inst_correct / num_reads
    inst_avg_h = inst_hamming / num_reads
    print(f"  Instance {inst+1:>2}: correct {inst_correct:>3}/{num_reads} ({inst_rate:>5.1f}%) | "
          f"Hamming {inst_avg_h:.1f} | best_E={ss.first.energy:.4f} target_E={opt_val_sa:.4f}")

elapsed = time.time() - t0
rate = 100.0 * total_correct / total_samples if total_samples > 0 else 0
avg_h = total_hamming / total_samples if total_samples > 0 else float('nan')

print(f"\n{'─' * 70}")
print(f"  총합: correct {total_correct}/{total_samples} ({rate:.1f}%) | Avg Hamming {avg_h:.1f} | {elapsed:.1f}s")

═══ SA 실험 (instances=10, reads=100, sweeps=5000) ═══
  QUBO: n=500, max_sub=14, α=0, coeff=lin20

  [Warning] 5000 clauses 후에도 유일성 미확보
  [에너지 분해] E_random=-334.0000, α·E_posiform=-0.0000, E_total=-334.0000
  Instance  1: correct   4/100 (  4.0%) | Hamming 6.0 | best_E=-334.0000 target_E=-334.0000
  [posiform] 3750 clauses로 유일성 확보
  [에너지 분해] E_random=-335.8000, α·E_posiform=-0.0000, E_total=-335.8000
  Instance  2: correct   0/100 (  0.0%) | Hamming 11.9 | best_E=-335.8000 target_E=-335.8000
  [posiform] 5000 clauses로 유일성 확보
  [에너지 분해] E_random=-371.0000, α·E_posiform=-0.0000, E_total=-371.0000
  Instance  3: correct   1/100 (  1.0%) | Hamming 13.9 | best_E=-371.0000 target_E=-371.0000
  [Warning] 5000 clauses 후에도 유일성 미확보
  [에너지 분해] E_random=-323.4000, α·E_posiform=-0.0000, E_total=-323.4000
  Instance  4: correct   1/100 (  1.0%) | Hamming 15.4 | best_E=-323.4000 target_E=-323.4000
  [Warning] 5000 clauses 후에도 유일성 미확보
  [에너지 분해] E_random=-323.9000, α·E_posiform=-0.0000, E_total=-323.90